# Data Lakes, Data Lakehouses and Modern Analytics Architecture - Companion Workbook

**Kajola Gbenga**
*Data Scientist & AI Engineer*

This notebook is the executable companion to the *Data Lakes, Data Lakehouses and Modern Analytics Architecture* handbook. Since this environment has no Spark cluster, no cloud storage account, and no `pyarrow`/`duckdb`/`delta-spark` installed, every concept below is demonstrated with a lightweight, from-scratch simulation built only from the Python standard library, `pandas`, and `sqlite3` - so every cell actually runs and produces real output.

What each simulation stands in for:

| In this notebook | In production (taught with real syntax in the handbook) |
|---|---|
| A local folder tree (`lake/bronze/...`) | Cloud object storage - S3 / ADLS Gen2 / GCS (Part 4) |
| CSV files | Parquet files (Part 7-8) |
| A hand-written JSON commit log | Delta Lake's `_delta_log` / Iceberg snapshots (Parts 12-14) |
| `sqlite3` | DuckDB / Trino / Spark SQL (Parts 16-18) |
| Plain Python merge logic | Delta/Iceberg/Hudi `MERGE` (Parts 13-15, 31) |

Every mechanic - medallion layering, partitioning, the small-file problem, compaction, schema evolution, CDC-style upserts, time travel, and incremental loading - behaves the same *way* here as in the real systems; only the storage engine underneath is swapped for something that runs anywhere.

In [1]:
import os, json, shutil, glob, sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

LAKE_ROOT = "/home/claude/lakehouse_nb/lake"
if os.path.exists(LAKE_ROOT):
    shutil.rmtree(LAKE_ROOT)
os.makedirs(LAKE_ROOT)

print("Local 'object storage' root created at:", LAKE_ROOT)

Local 'object storage' root created at: /home/claude/lakehouse_nb/lake


## 1. Generate Realistic Source Data

A small synthetic e-commerce `orders` stream - the same conceptual schema used throughout the handbook - arriving as a series of small batches over several days, which is exactly the pattern that creates the small-file problem (Part 10) if landed naively.

In [2]:
rng = np.random.default_rng(7)

customers = [f"CUST{1000+i}" for i in range(200)]
regions = ["North", "South", "East", "West"]
statuses = ["pending", "completed", "cancelled", "refunded"]

def generate_batch(batch_date, n_rows, start_id):
    rows = []
    for i in range(n_rows):
        rows.append({
            "order_id": start_id + i,
            "customer_id": rng.choice(customers),
            "order_date": batch_date,
            "region": rng.choice(regions),
            "order_status": rng.choice(statuses, p=[0.05, 0.85, 0.06, 0.04]),
            "amount": round(float(rng.uniform(5, 500)), 2),
            "_ingested_at": batch_date + " 0" + str(rng.integers(0,9)) + ":00:00",
        })
    return pd.DataFrame(rows)

print("Batch generator ready.")

Batch generator ready.


## 2. Bronze Layer: Partitioned Landing, and the Small File Problem (Parts 6, 9, 10)

Each small batch is written as its own file under a `order_date=YYYY-MM-DD/` partition - exactly how a naive streaming writer with a very short trigger interval behaves (Part 30). We then inspect the file count and average file size to see the small-file problem directly, the same way you would diagnose it against real Parquet files in production.

In [3]:
BRONZE_PATH = f"{LAKE_ROOT}/bronze/orders"
os.makedirs(BRONZE_PATH, exist_ok=True)

next_id = 1
dates = [(datetime(2026, 9, 1) + timedelta(days=d)).strftime('%Y-%m-%d') for d in range(5)]

for d in dates:
    partition_dir = f"{BRONZE_PATH}/order_date={d}"
    os.makedirs(partition_dir, exist_ok=True)
    # simulate MANY small streaming micro-batches landing throughout the day (small-file problem)
    for batch_num in range(20):
        batch_df = generate_batch(d, n_rows=rng.integers(3, 12), start_id=next_id)
        next_id += len(batch_df)
        batch_df.to_csv(f"{partition_dir}/part-{batch_num:04d}.csv", index=False)

all_files = glob.glob(f"{BRONZE_PATH}/**/*.csv", recursive=True)
sizes_kb = [os.path.getsize(f) / 1024 for f in all_files]
print(f"Bronze files written: {len(all_files)}")
print(f"Average file size: {np.mean(sizes_kb):.2f} KB (target for a healthy table is usually 128MB-1GB - Part 10)")
print(f"Total rows landed: {next_id - 1}")

Bronze files written: 100
Average file size: 0.52 KB (target for a healthy table is usually 128MB-1GB - Part 10)
Total rows landed: 700


In [4]:
# Diagnose the small file problem per partition (Part 10 / Part 73 troubleshooting pattern)
partition_stats = []
for d in dates:
    files = glob.glob(f"{BRONZE_PATH}/order_date={d}/*.csv")
    partition_stats.append({"order_date": d, "file_count": len(files),
                             "avg_size_kb": round(np.mean([os.path.getsize(f)/1024 for f in files]), 2)})
display(pd.DataFrame(partition_stats))
print("\n20 tiny files per day is exactly the pattern that overwhelms object storage listing and query planning at scale.")


20 tiny files per day is exactly the pattern that overwhelms object storage listing and query planning at scale.


,order_date,file_count,avg_size_kb
0,2026-09-01,20,0.53
1,2026-09-02,20,0.58
2,2026-09-03,20,0.50
3,2026-09-04,20,0.47
4,2026-09-05,20,0.53


## 3. Compaction (Part 10, 13): Rewriting Small Files into Fewer, Larger Ones

The same operation Delta Lake's `OPTIMIZE` performs internally: read all the small files in a partition, concatenate them, and rewrite as one (or a few) larger files.

In [5]:
def compact_partition(partition_dir):
    files = glob.glob(f"{partition_dir}/*.csv")
    if len(files) <= 1:
        return 0, 0
    dfs = [pd.read_csv(f) for f in files]
    combined = pd.concat(dfs, ignore_index=True)
    for f in files:
        os.remove(f)
    combined.to_csv(f"{partition_dir}/part-compacted-0000.csv", index=False)
    return len(files), 1

before_total = len(glob.glob(f"{BRONZE_PATH}/**/*.csv", recursive=True))
for d in dates:
    compact_partition(f"{BRONZE_PATH}/order_date={d}")
after_total = len(glob.glob(f"{BRONZE_PATH}/**/*.csv", recursive=True))

print(f"Files before compaction: {before_total}")
print(f"Files after compaction:  {after_total}")
print(f"Reduction: {100*(1 - after_total/before_total):.1f}%")

Files before compaction: 100
Files after compaction:  5
Reduction: 95.0%


## 4. Partition Pruning (Part 9): Why Filtering on the Partition Column Matters

Simulating the difference between a pruned read (only opens the relevant partition's files) and a naive full scan (opens every file), by counting files actually touched.

In [6]:
def read_with_partition_pruning(base_path, order_date_filter=None):
    if order_date_filter:
        pattern = f"{base_path}/order_date={order_date_filter}/*.csv"
    else:
        pattern = f"{base_path}/**/*.csv"
    files = glob.glob(pattern, recursive=True)
    dfs = [pd.read_csv(f) for f in files]
    return pd.concat(dfs, ignore_index=True), len(files)

# Full scan: every partition opened
full_scan_df, full_scan_files = read_with_partition_pruning(BRONZE_PATH)

# Pruned: only one partition opened, because the query filters on order_date
pruned_df, pruned_files = read_with_partition_pruning(BRONZE_PATH, order_date_filter="2026-09-03")

print(f"Full scan touched {full_scan_files} files and read {len(full_scan_df)} rows")
print(f"Pruned read (WHERE order_date = '2026-09-03') touched {pruned_files} file(s) and read {len(pruned_df)} rows")
print(f"\nFiles skipped by pruning: {full_scan_files - pruned_files} out of {full_scan_files}")

Full scan touched 5 files and read 700 rows
Pruned read (WHERE order_date = '2026-09-03') touched 1 file(s) and read 132 rows

Files skipped by pruning: 4 out of 5


## 5. Bronze to Silver: Data Quality Gate (Parts 6, 24)

Clean, validate, and deduplicate bronze data before promoting it to silver - invalid rows are quarantined rather than silently dropped, exactly as the handbook's data-quality-gate pattern recommends.

In [7]:
bronze_full, _ = read_with_partition_pruning(BRONZE_PATH)

def data_quality_report(df):
    return {
        "total_rows": len(df),
        "null_customer_id": int(df["customer_id"].isna().sum()),
        "duplicate_order_id": int(df["order_id"].duplicated().sum()),
        "negative_amount": int((df["amount"] < 0).sum()),
        "invalid_status": int((~df["order_status"].isin(statuses)).sum()),
    }

report = data_quality_report(bronze_full)
display(pd.DataFrame([report]))

# Apply the quality gate: valid rows -> silver, invalid rows -> quarantine (Part 24, 73)
valid_mask = (
    bronze_full["customer_id"].notna()
    & (~bronze_full["order_id"].duplicated(keep="first"))
    & (bronze_full["amount"] >= 0)
    & (bronze_full["order_status"].isin(statuses))
)
silver_df = bronze_full[valid_mask].copy()
quarantined_df = bronze_full[~valid_mask].copy()

print(f"\nPromoted to silver: {len(silver_df)} rows")
print(f"Quarantined: {len(quarantined_df)} rows")


Promoted to silver: 700 rows
Quarantined: 0 rows


,total_rows,null_customer_id,duplicate_order_id,negative_amount,invalid_status
0,700,0,0,0,0


In [8]:
SILVER_PATH = f"{LAKE_ROOT}/silver/orders"
os.makedirs(SILVER_PATH, exist_ok=True)
silver_df.to_csv(f"{SILVER_PATH}/orders.csv", index=False)
print("Silver table written:", f"{SILVER_PATH}/orders.csv")
display(silver_df.head())

Silver table written: /home/claude/lakehouse_nb/lake/silver/orders/orders.csv


,order_id,customer_id,order_date,region,order_status,amount,_ingested_at
0,213,CUST1010,2026-09-02,West,completed,235.07,2026-09-02 01:00:00
1,214,CUST1045,2026-09-02,South,completed,144.04,2026-09-02 03:00:00
2,215,CUST1006,2026-09-02,South,completed,131.64,2026-09-02 02:00:00
3,216,CUST1042,2026-09-02,South,completed,206.31,2026-09-02 01:00:00
4,217,CUST1177,2026-09-02,East,completed,369.30,2026-09-02 08:00:00


## 6. A Minimal Transaction Log: Simulating Delta Lake / Iceberg (Parts 12-14)

A tiny, from-scratch "table format": every write appends a new, timestamped snapshot to a JSON commit log alongside a versioned CSV file, instead of overwriting data in place. This is a simplified stand-in for Delta Lake's `_delta_log` (Part 13) or Iceberg's snapshot/manifest chain (Part 14) - enough to demonstrate **atomic commits** and **time travel** end to end.

In [9]:
class MiniLakehouseTable:
    """A minimal, from-scratch simulation of an ACID table format (Delta/Iceberg-style)."""

    def __init__(self, table_path):
        self.table_path = table_path
        self.log_path = f"{table_path}/_commit_log.json"
        os.makedirs(table_path, exist_ok=True)
        if not os.path.exists(self.log_path):
            with open(self.log_path, "w") as f:
                json.dump([], f)

    def _log(self):
        with open(self.log_path) as f:
            return json.load(f)

    def _write_log(self, entries):
        with open(self.log_path, "w") as f:
            json.dump(entries, f, indent=2)

    def commit(self, df, operation):
        entries = self._log()
        version = len(entries)
        data_file = f"{self.table_path}/v{version}.csv"
        df.to_csv(data_file, index=False)
        entries.append({
            "version": version,
            "operation": operation,
            "timestamp": datetime.now().isoformat(),
            "data_file": data_file,
            "row_count": len(df),
        })
        self._write_log(entries)
        print(f"Committed version {version} ({operation}): {len(df)} rows")
        return version

    def read(self, version=None):
        entries = self._log()
        if not entries:
            raise ValueError("Table has no committed versions yet")
        target = entries[version] if version is not None else entries[-1]
        return pd.read_csv(target["data_file"])

    def history(self):
        return pd.DataFrame(self._log())[["version", "operation", "timestamp", "row_count"]]

print("MiniLakehouseTable defined - a simplified stand-in for Delta Lake / Iceberg's ACID commit log.")

MiniLakehouseTable defined - a simplified stand-in for Delta Lake / Iceberg's ACID commit log.


In [10]:
silver_table = MiniLakehouseTable(f"{LAKE_ROOT}/silver_delta/orders")

# Initial commit (like a CREATE TABLE + first write)
silver_table.commit(silver_df, operation="INITIAL_LOAD")

display(silver_table.history())

Committed version 0 (INITIAL_LOAD): 700 rows


,version,operation,timestamp,row_count
0,0,INITIAL_LOAD,2026-09-16T22:16:59.185982,700


## 7. CDC-Style MERGE (Upsert) - Parts 13, 31

Applying a batch of Change Data Capture events (`c` = create, `u` = update, `d` = delete) into the table via a `MERGE`, exactly like Delta/Iceberg's `MERGE INTO ... WHEN MATCHED ... WHEN NOT MATCHED` pattern (Part 13.2), producing a new, atomically-committed version.

In [11]:
def apply_cdc_merge(table: MiniLakehouseTable, cdc_events: pd.DataFrame, key="order_id"):
    current = table.read()
    deletes = cdc_events[cdc_events["op"] == "d"][key]
    current = current[~current[key].isin(deletes)]

    upserts = cdc_events[cdc_events["op"].isin(["c", "u"])].drop(columns=["op"])
    current = pd.concat([current[~current[key].isin(upserts[key])], upserts], ignore_index=True)

    return table.commit(current, operation="MERGE (CDC)")

# Simulate a CDC batch: 3 status updates and 1 cancellation-as-delete
sample_ids = silver_df["order_id"].sample(4, random_state=1).tolist()
cdc_batch = silver_df[silver_df["order_id"].isin(sample_ids)].copy()
cdc_batch["op"] = ["u", "u", "u", "d"]
cdc_batch.loc[cdc_batch["op"] == "u", "order_status"] = "completed"

print("CDC batch to apply:")
display(cdc_batch[["order_id", "op", "order_status"]])

apply_cdc_merge(silver_table, cdc_batch)
display(silver_table.history())

CDC batch to apply:
Committed version 1 (MERGE (CDC)): 699 rows


,order_id,op,order_status
329,118,u,completed
620,336,u,completed
626,342,u,completed
681,304,d,refunded


,version,operation,timestamp,row_count
0,0,INITIAL_LOAD,2026-09-16T22:16:59.185982,700
1,1,MERGE (CDC),2026-09-16T22:16:59.194525,699


## 8. Time Travel (Parts 13-14)

Because every commit is preserved rather than overwritten, the table can be read as of any prior version - the same capability as `VERSION AS OF` in Delta Lake or a snapshot ID in Iceberg.

In [12]:
v0 = silver_table.read(version=0)
v_latest = silver_table.read()

print(f"Version 0 (initial load): {len(v0)} rows")
print(f"Latest version (after CDC merge): {len(v_latest)} rows")
print(f"\nOne of the deleted order_ids, as it looked in version 0:")
deleted_id = cdc_batch[cdc_batch['op']=='d']['order_id'].iloc[0]
display(v0[v0['order_id'] == deleted_id])
print(f"The same order_id in the latest version (should be gone - it was deleted by CDC):")
display(v_latest[v_latest['order_id'] == deleted_id])

Version 0 (initial load): 700 rows
Latest version (after CDC merge): 699 rows

One of the deleted order_ids, as it looked in version 0:
The same order_id in the latest version (should be gone - it was deleted by CDC):


,order_id,customer_id,order_date,region,order_status,amount,_ingested_at
681,304,CUST1020,2026-09-03,North,refunded,304.44,2026-09-03 03:00:00


,order_id,customer_id,order_date,region,order_status,amount,_ingested_at


## 9. Schema Evolution (Part 26)

Adding a new, nullable column is a safe, additive schema change - old rows simply get a null value for it, and old readers that don't know about the column are unaffected. This mirrors Delta Lake's `mergeSchema` option and Iceberg's column-ID-based evolution.

In [13]:
# Safe evolution: add a new nullable column, "payment_method"
latest = silver_table.read()
latest["payment_method"] = rng.choice(["card", "transfer", "cash"], size=len(latest))

new_version = silver_table.commit(latest, operation="SCHEMA_EVOLUTION (added payment_method)")
display(silver_table.history())

# Old version still reads fine, WITHOUT the new column - proving old and new versions can coexist
old = silver_table.read(version=0)
print("\nColumns in version 0 (before evolution):", list(old.columns))
print("Columns in the latest version (after evolution):", list(latest.columns))

Committed version 2 (SCHEMA_EVOLUTION (added payment_method)): 699 rows

Columns in version 0 (before evolution): ['order_id', 'customer_id', 'order_date', 'region', 'order_status', 'amount', '_ingested_at']
Columns in the latest version (after evolution): ['order_id', 'customer_id', 'order_date', 'region', 'order_status', 'amount', '_ingested_at', 'payment_method']


,version,operation,timestamp,row_count
0,0,INITIAL_LOAD,2026-09-16T22:16:59.185982,700
1,1,MERGE (CDC),2026-09-16T22:16:59.194525,699
2,2,SCHEMA_EVOLUTION (added payment_method),2026-09-16T22:16:59.207014,699


## 10. Gold Layer: Business-Ready KPI Table (Part 6)

Aggregating the (now schema-evolved, CDC-merged) silver table into a gold, dashboard-ready KPI table - the same shape of query taught throughout the companion SQL handbook, now sitting on top of the lakehouse layer.

In [14]:
current_silver = silver_table.read()
completed = current_silver[current_silver["order_status"] == "completed"]

gold_daily_revenue = (
    completed.groupby(["order_date", "region"])
    .agg(revenue=("amount", "sum"), orders=("order_id", "nunique"))
    .reset_index()
    .sort_values(["order_date", "region"])
)

GOLD_PATH = f"{LAKE_ROOT}/gold/daily_revenue_by_region"
os.makedirs(GOLD_PATH, exist_ok=True)
gold_daily_revenue.to_csv(f"{GOLD_PATH}/daily_revenue_by_region.csv", index=False)

display(gold_daily_revenue)
print(f"\nGold table written to {GOLD_PATH}")


Gold table written to /home/claude/lakehouse_nb/lake/gold/daily_revenue_by_region


,order_date,region,revenue,orders
0,2026-09-01,East,5716.44,23
1,2026-09-01,North,6121.65,23
2,2026-09-01,South,8097.83,32
3,2026-09-01,West,9992.65,37
4,2026-09-02,East,8233.74,32
5,2026-09-02,North,7204.17,27
6,2026-09-02,South,10297.61,46
7,2026-09-02,West,7031.07,30
8,2026-09-03,East,7494.29,27
9,2026-09-03,North,6644.09,28


## 11. Incremental Processing with a High-Water Mark (Part 32)

Rather than reprocessing the entire silver table on every run, track the maximum `_ingested_at` value already processed, and only pull rows newer than that mark on the next run - the standard incremental-load pattern.

In [15]:
# Simulate an ETL control table tracking the high-water mark
control_table = {"orders": current_silver["_ingested_at"].max()}
print(f"High-water mark after this run: {control_table['orders']}")

# Simulate new data arriving after the mark
new_batch = generate_batch("2026-09-06", n_rows=15, start_id=int(current_silver['order_id'].max()) + 1)
new_batch["payment_method"] = rng.choice(["card", "transfer", "cash"], size=len(new_batch))

incremental_rows = new_batch[new_batch["_ingested_at"] > control_table["orders"]]
print(f"New rows to process incrementally (only what's past the watermark): {len(incremental_rows)}")

# Advance the watermark after processing
control_table["orders"] = max(control_table["orders"], incremental_rows["_ingested_at"].max())
print(f"Watermark advanced to: {control_table['orders']}")

High-water mark after this run: 2026-09-05 08:00:00
New rows to process incrementally (only what's past the watermark): 15
Watermark advanced to: 2026-09-06 07:00:00


## 12. Querying the Lake with SQL (Standing in for DuckDB / Trino - Parts 17-18)

`sqlite3` is used here purely as a runnable SQL engine over the gold CSV output - conceptually the same role DuckDB plays when it runs `SELECT * FROM 'gold.parquet'` directly against lake files (Part 17), or Trino plays federating a query across the lake and an operational database (Part 18).

In [16]:
con = sqlite3.connect(":memory:")
gold_daily_revenue.to_sql("gold_daily_revenue_by_region", con, index=False, if_exists="replace")

result = pd.read_sql_query("""
    SELECT region, SUM(revenue) AS total_revenue, SUM(orders) AS total_orders,
           ROUND(SUM(revenue) * 1.0 / SUM(orders), 2) AS avg_order_value
    FROM gold_daily_revenue_by_region
    GROUP BY region
    ORDER BY total_revenue DESC
""", con)

display(result)

,region,total_revenue,total_orders,avg_order_value
0,West,40701.60,163,249.70
1,South,38516.60,161,239.23
2,North,34880.74,136,256.48
3,East,34113.67,133,256.49


## 13. Data Observability Check (Part 41)

A minimal freshness + volume health check, the same pattern a scheduled monitoring job would run against a real production table.

In [17]:
def check_table_health(df, timestamp_col, expected_min_rows=10, max_staleness_hours=48):
    latest_ts = pd.to_datetime(df[timestamp_col]).max()
    freshness_hours = (pd.Timestamp.now() - latest_ts).total_seconds() / 3600
    return {
        "row_count": len(df),
        "row_count_ok": len(df) >= expected_min_rows,
        "freshness_hours": round(freshness_hours, 1),
        "freshness_ok": freshness_hours <= max_staleness_hours,
    }

health = check_table_health(current_silver, "_ingested_at")
display(pd.DataFrame([health]))
print("\nIn production this check would run on a schedule (Part 42) and alert (Part 41) on failure.")


In production this check would run on a schedule (Part 42) and alert (Part 41) on failure.


,row_count,row_count_ok,freshness_hours,freshness_ok
0,699,True,278.3,False


## 14. Summary

This workbook executed a complete, from-scratch simulation of a lakehouse pipeline: partitioned bronze landing, the small-file problem and compaction, partition pruning, a data-quality gate promoting bronze to silver, an ACID-style commit log with time travel, a CDC `MERGE` upsert, safe schema evolution, a gold KPI aggregation, incremental loading via a high-water mark, SQL-based querying, and a basic observability check.

Every one of these mechanics is implemented for real, at production scale, by the technologies taught throughout the handbook - Parquet (Part 7-8) instead of CSV, Delta Lake / Iceberg / Hudi (Parts 12-15) instead of the hand-written commit log, Spark / DuckDB / Trino (Parts 16-18, 35) instead of plain pandas and `sqlite3`, and a real orchestrator (Part 42) instead of manually running cells in order. The *shape* of every operation carries over directly.

**Kajola Gbenga**
*Data Scientist & AI Engineer*